In [2]:
import json
import os

In [ ]:
# load data
all_data = json.load(open('../C-subreddit_selection/comm_data.json'))

# inspect data
print(len(all_data))
print(list(all_data.keys())[0])
# print(all_data["worldnews"].keys()) # dict_keys(['subreddit_info', 'monthly_record'])
print(all_data["worldnews"]["subreddit_info"])
# print(all_data["worldnews"]["monthly_record"])

for subreddit_name, subreddit_all in all_data.items():
    if subreddit_all["subreddit_info"]["type"] != "topic":
        print(subreddit_name)
        print(subreddit_all["subreddit_info"].keys())
        print(subreddit_all["subreddit_info"]["info"]) 
        break
    else:
        print(subreddit_name)

print(all_data["worldnews"]["monthly_record"].keys())
print(all_data["worldnews"]["monthly_record"]['2020-07'][0].keys())
['submission_id', 'submission_date', 'news_title', 'news_content', 'news_keywords', 'reactions']

for entry in all_data["worldnews"]["monthly_record"]['2020-07']:
    print(entry["submission_date"]) # 2020-07-01, 02, etc
    print(entry["news_keywords"])   # ['countries', 'italy', 'spain', 'tedros adhanom ghebreyesus', 'world health organization', 'disease', 'covid-19 pandemic', 'epidemic peaks', 'coronavirus pandemic', 'combination']

207
worldnews
{'type': 'topic', 'info': {'Geo_Estimate': 'GLOBAL', 'Topic': [['SOCIAL_DISCUSSION_AND_THOUGHTS']]}}
worldnews
collapse
news
Sino
australia
dict_keys(['type', 'info'])
{'geo_path': ['GLOBAL-OCEANIA-AUS'], 'is_covid_specific': False}
dict_keys(['2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09', '2020-10', '2020-11', '2020-12'])
dict_keys(['submission_id', 'submission_date', 'news_title', 'news_content', 'news_keywords', 'reactions'])
2020-07-01
['countries', 'italy', 'spain', 'tedros adhanom ghebreyesus', 'world health organization', 'disease', 'covid-19 pandemic', 'epidemic peaks', 'coronavirus pandemic', 'combination']
2020-07-01
['drugs', 'people', 'uk patients', 'remdesivir', 'countries', 'coronavirus', 'accessible video player', 'treatments', 'part', 'europe']
2020-07-01
['drugs', 'remdesivir', 'patient', 'health', 'gilead deal', 'pricing structure', 'coronavirus patients', 'entire global supply', 'results', 'covid-19']


In [12]:
from collections import defaultdict

# counting (keyword, subreddit, month, date)
keyword_counts = defaultdict(
    lambda: defaultdict(
        lambda: defaultdict(
            lambda: defaultdict(int)
        )
    )
)
# while saving a keyword-to-[(subreddit, month, submission_id)'s] lookup
keyword_lookup = defaultdict(list)

for subreddit, sub_data in all_data.items():
    for month, entries in sub_data["monthly_record"].items():
        for entry in entries:
            date_str = entry["submission_date"] # format: YYYY-MM-DD
            assert len(date_str) == 10
            assert month == date_str[:7]
            date = date_str[8:]
            # print(subreddit, month, date_str)
            for kw in entry.get("news_keywords", []):
                # print("\t", kw)
                if len(kw.split()) > 1:
                    keyword_counts[kw][subreddit][month][date] += 1
                    keyword_lookup[kw].append((subreddit, month, entry["submission_id"]))


In [ ]:

flat_rows = [
    (kw, sub, month, day, count)
    for kw in keyword_counts
    for sub in keyword_counts[kw]
    for month in keyword_counts[kw][sub]
    for day, count in keyword_counts[kw][sub][month].items()
]

# load into a pandas dataframe and save
import pandas as pd
df = pd.DataFrame(flat_rows, columns=["keyword", "subreddit", "month", "day", "count"])       
df.to_csv("keyword_spread.csv", index=False)

# save keyword lookup
with open("keyword_lookup.json", "w") as f:
    json.dump(keyword_lookup, f, indent=4)